# Controlled Measurement Extraction and Validation

**Stage 03 — Curation, Quality Control and Reproducible Analysis of Collagen-Hydrogel Rheology Data**

This notebook converts the audited collagen-hydrogel rheology workbooks into a
canonical, provenance-preserving measurement table. It builds directly on the
completed Stage 01 audit and Stage 02 schema design.

The source dataset is the Zenodo collagen-hydrogel rheology dataset
(DOI `10.5281/zenodo.17413651`), extracted from `Reología.rar`, containing
frequency- and time-sweep measurements for bovine collagen hydrogels at
0.8, 1.5 and 2.3 mg/mL, with testing reported at 37 °C.

The workflow is intentionally conservative:

- the 20 source workbooks are treated as immutable;
- one processed record is created for each audited measurement row;
- `experiment_id`, `source_excel_row` and deterministic `measurement_id` values
  preserve row-level provenance;
- approved unit-label mappings standardise labels only — numerical measurements
  are not converted or silently corrected;
- frequency-sweep fields that are structurally absent remain missing rather than
  being filled with zeros;
- the documented `G′ = 0 Pa` observation is retained and linked to its Stage 01
  quality-control finding;
- the time-sweep `Meas. Pts.` field is retained as a measurement-point index and
  is **not** interpreted as elapsed time; and
- matching concentration and replicate labels across frequency- and time-sweep
  workbooks are **not** treated as evidence that they represent the same
  physical hydrogel specimen.

The notebook is written to support **Kernel → Restart & Run All**. Required
control tables are reloaded from disk at the start, helper functions are defined
before use, and file-writing operations are deterministic and repeatable.

<details>
<summary><strong>Rheology variables used in this dataset</strong></summary>

- **Angular frequency** describes how quickly the oscillatory deformation is
  applied.
- **Storage modulus, G′** describes the more elastic or solid-like component of
  the hydrogel response.
- **Loss modulus, G″** describes the more viscous or liquid-like component.
- **Temperature** records the measurement condition.
- **Shear stress**, **strain** and **complex viscosity** are present in the
  time-sweep source structure but not in the frequency-sweep source structure.

These definitions support interpretation of the curated fields; they do not
replace experiment-specific rheological analysis.

</details>


In [2]:
# ================================================================
# STAGE 03 — REPRODUCIBLE SETUP AND INPUT PREFLIGHT
# ================================================================

from pathlib import Path
from importlib.metadata import PackageNotFoundError, version
import re
import sys


# ----------------------------------------------------------------
# 1. Verify the intended project environment before pandas import
# ----------------------------------------------------------------

EXPECTED_PYTHON = "3.11"
EXPECTED_NUMPY = "1.26"
EXPECTED_PANDAS = "2.2"

def installed_version(package_name):
    try:
        return version(package_name)
    except PackageNotFoundError:
        return None


python_version = (
    f"{sys.version_info.major}."
    f"{sys.version_info.minor}."
    f"{sys.version_info.micro}"
)

numpy_version = installed_version("numpy")
pandas_version = installed_version("pandas")
openpyxl_version = installed_version("openpyxl")
environment_name = Path(sys.prefix).name

environment_problems = []

if not python_version.startswith(EXPECTED_PYTHON):
    environment_problems.append(
        f"Python {python_version} detected; expected {EXPECTED_PYTHON}.x."
    )

if numpy_version is None:
    environment_problems.append("NumPy is not installed.")
elif not numpy_version.startswith(EXPECTED_NUMPY):
    environment_problems.append(
        f"NumPy {numpy_version} detected; expected {EXPECTED_NUMPY}.x."
    )

if pandas_version is None:
    environment_problems.append("pandas is not installed.")
elif not pandas_version.startswith(EXPECTED_PANDAS):
    environment_problems.append(
        f"pandas {pandas_version} detected; expected {EXPECTED_PANDAS}.x."
    )

if openpyxl_version is None:
    environment_problems.append("openpyxl is not installed.")

if environment_problems:
    raise RuntimeError(
        "PROJECT ENVIRONMENT CHECK FAILED\n\n"
        + "\n".join(f"- {problem}" for problem in environment_problems)
        + "\n\nSelect the project kernel 'Python (hydrogel-rheology)' "
          "and restart the kernel."
    )


import pandas as pd
from openpyxl import load_workbook
from IPython.display import display


# ----------------------------------------------------------------
# 2. Locate the project root reproducibly
# ----------------------------------------------------------------

current_directory = Path.cwd().resolve()

project_root = None

for candidate in [current_directory, *current_directory.parents]:
    if (
        (candidate / "metadata" / "experiment_register.csv").exists()
        and
        (
            candidate
            / "metadata"
            / "measurement_data_dictionary.csv"
        ).exists()
        and
        (
            candidate
            / "data"
            / "quality_control"
            / "data_quality_issue_log.csv"
        ).exists()
    ):
        project_root = candidate
        break

if project_root is None:
    raise FileNotFoundError(
        "Could not locate the collagen-hydrogel project root.\n\n"
        "Expected to find the Stage 01/02 metadata and quality-control "
        "files in the project directory."
    )


# ----------------------------------------------------------------
# 3. Define required Stage 01 / Stage 02 inputs
# ----------------------------------------------------------------

experiment_register_file = (
    project_root / "metadata" / "experiment_register.csv"
)

measurement_dictionary_file = (
    project_root / "metadata" / "measurement_data_dictionary.csv"
)

workbook_qc_file = (
    project_root / "metadata" / "workbook_schema_qc.csv"
)

issue_log_file = (
    project_root
    / "data"
    / "quality_control"
    / "data_quality_issue_log.csv"
)

unit_mapping_file = (
    project_root
    / "data"
    / "quality_control"
    / "unit_label_mapping_audit.csv"
)

source_directory = (
    project_root
    / "data"
    / "interim"
    / "zenodo_17413651_v1"
)

required_paths = {
    "experiment register": experiment_register_file,
    "measurement data dictionary": measurement_dictionary_file,
    "workbook schema QC": workbook_qc_file,
    "data-quality issue log": issue_log_file,
    "unit-label mapping audit": unit_mapping_file,
    "source workbook directory": source_directory,
}

missing_paths = [
    f"{label}: {path}"
    for label, path in required_paths.items()
    if not path.exists()
]

if missing_paths:
    raise FileNotFoundError(
        "Required Stage 01/02 input(s) are missing:\n\n"
        + "\n".join(f"- {item}" for item in missing_paths)
    )


# ----------------------------------------------------------------
# 4. Reload control tables from disk
# ----------------------------------------------------------------

experiment_register = pd.read_csv(
    experiment_register_file,
    encoding="utf-8-sig",
    dtype={"replicate_id": "string"},
)

measurement_dictionary = pd.read_csv(
    measurement_dictionary_file,
    encoding="utf-8-sig",
)

workbook_qc = pd.read_csv(
    workbook_qc_file,
    encoding="utf-8-sig",
    dtype={"replicate_id": "string"},
)

issue_log = pd.read_csv(
    issue_log_file,
    encoding="utf-8-sig",
)

unit_mapping = pd.read_csv(
    unit_mapping_file,
    encoding="utf-8-sig",
)

unit_mapping["mapping_success_bool"] = (
    unit_mapping["mapping_success"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("true")
)

measurement_schema_fields = (
    measurement_dictionary["field_name"]
    .astype(str)
    .tolist()
)


# ----------------------------------------------------------------
# 5. Define the audited source layouts
# ----------------------------------------------------------------

SOURCE_LAYOUTS = {
    "frequency_sweep": {
        "headers": [
            "Meas. Pts.",
            "Angular Frequency",
            "Storage Modulus",
            "Loss Modulus",
            "Temperature",
        ],
        "units": [
            None,
            "[rad/s]",
            "[Pa]",
            "[Pa]",
            "[°C]",
        ],
    },
    "time_sweep": {
        "headers": [
            "Meas. Pts.",
            "Shear Stress",
            "Angular Frequency",
            "Storage Modulus",
            "Strain",
            "Complex Viscosity",
            "Loss Modulus",
            "Temperature",
        ],
        "units": [
            None,
            "[Pa]",
            "[rad/s]",
            "[Pa]",
            "[%]",
            "[Pa·s]",
            "[Pa]",
            "[°C]",
        ],
    },
}


# ----------------------------------------------------------------
# 6. Shared restart-safe helper functions
# ----------------------------------------------------------------

def require_objects(*names):
    """Raise a clear error if a cell is run before its prerequisites."""
    missing = [name for name in names if name not in globals()]
    if missing:
        raise RuntimeError(
            "Required upstream notebook object(s) are missing:\n\n"
            + "\n".join(f"- {name}" for name in missing)
            + "\n\nRun the notebook from the top using Restart & Run All."
        )


def audited_row_count(filename):
    """Return the single Stage 01 audited measurement-row count."""
    match = workbook_qc.loc[
        workbook_qc["original_filename"].eq(filename),
        "data_row_count",
    ]

    if len(match) != 1:
        raise ValueError(
            f"Expected exactly one Stage 01 workbook-QC record for "
            f"{filename}, but found {len(match)}."
        )

    return int(match.iloc[0])


def resolve_unit_label(
    filename,
    worksheet,
    variable,
    observed_unit,
    expected_unit,
):
    """
    Validate one source unit label.

    Returns (status, standardised_unit). Numerical values are never changed.
    """
    if expected_unit is None:
        if observed_unit is None:
            return "no_unit_expected", None
        return "unexpected_unit_label", observed_unit

    if observed_unit == expected_unit:
        return "canonical", expected_unit

    mapping_match = unit_mapping.loc[
        unit_mapping["original_filename"].eq(filename)
        & unit_mapping["worksheet"].eq(worksheet)
        & unit_mapping["variable"].eq(variable)
        & unit_mapping["observed_unit"].eq(observed_unit)
    ]

    if (
        len(mapping_match) == 1
        and bool(mapping_match["mapping_success_bool"].iloc[0])
        and mapping_match["standardized_unit"].iloc[0] == expected_unit
    ):
        return "approved_mapping", expected_unit

    return "unresolved", observed_unit


def extract_registered_experiment(experiment):
    """
    Extract one registered workbook into the Stage 02 canonical schema.

    The source workbook is opened read-only and is never modified.
    """
    filename = experiment["original_filename"]
    sheet_name = experiment["sheet_name"]
    experiment_id = experiment["experiment_id"]
    test_type = experiment["test_type_standard"]

    if test_type not in SOURCE_LAYOUTS:
        raise ValueError(
            f"Unexpected test_type_standard '{test_type}' in {filename}."
        )

    layout = SOURCE_LAYOUTS[test_type]
    expected_headers = layout["headers"]
    expected_units = layout["units"]
    max_source_column = len(expected_headers)

    workbook_path = source_directory / filename

    if not workbook_path.exists():
        raise FileNotFoundError(
            f"Registered source workbook was not found:\n{workbook_path}"
        )

    workbook = load_workbook(
        workbook_path,
        read_only=True,
        data_only=False,
        keep_links=False,
    )

    try:
        if sheet_name not in workbook.sheetnames:
            raise ValueError(
                f"Worksheet '{sheet_name}' was not found in {filename}."
            )

        worksheet = workbook[sheet_name]

        source_headers = list(
            next(
                worksheet.iter_rows(
                    min_row=4,
                    max_row=4,
                    min_col=1,
                    max_col=max_source_column,
                    values_only=True,
                )
            )
        )

        source_units = list(
            next(
                worksheet.iter_rows(
                    min_row=5,
                    max_row=5,
                    min_col=1,
                    max_col=max_source_column,
                    values_only=True,
                )
            )
        )

        if source_headers != expected_headers:
            raise ValueError(
                f"Source heading mismatch in {filename} / {sheet_name}.\n\n"
                f"Expected: {expected_headers}\n"
                f"Found:    {source_headers}"
            )

        extracted_records = []
        measurement_rows = []

        for source_excel_row, values in enumerate(
            worksheet.iter_rows(
                min_row=6,
                max_row=worksheet.max_row,
                min_col=1,
                max_col=max_source_column,
                values_only=True,
            ),
            start=6,
        ):
            if values[0] is None:
                continue

            measurement_rows.append(source_excel_row)

            if test_type == "frequency_sweep":
                record = {
                    "measurement_id": (
                        f"{experiment_id}::row_{source_excel_row}"
                    ),
                    "experiment_id": experiment_id,
                    "source_excel_row": source_excel_row,
                    "measurement_point": values[0],
                    "angular_frequency_rad_s": values[1],
                    "storage_modulus_pa": values[2],
                    "loss_modulus_pa": values[3],
                    "temperature_c": values[4],
                    "shear_stress_pa": pd.NA,
                    "strain_percent": pd.NA,
                    "complex_viscosity_pa_s": pd.NA,
                }

            else:
                record = {
                    "measurement_id": (
                        f"{experiment_id}::row_{source_excel_row}"
                    ),
                    "experiment_id": experiment_id,
                    "source_excel_row": source_excel_row,
                    "measurement_point": values[0],
                    "angular_frequency_rad_s": values[2],
                    "storage_modulus_pa": values[3],
                    "loss_modulus_pa": values[6],
                    "temperature_c": values[7],
                    "shear_stress_pa": values[1],
                    "strain_percent": values[4],
                    "complex_viscosity_pa_s": values[5],
                }

            extracted_records.append(record)

        extra_source_columns = max(
            int(worksheet.max_column) - max_source_column,
            0,
        )

    finally:
        workbook.close()

    measurements = (
        pd.DataFrame(extracted_records)
        .reindex(columns=measurement_schema_fields)
    )

    unit_records = []

    for variable, observed_unit, expected_unit in zip(
        expected_headers,
        source_units,
        expected_units,
    ):
        unit_status, standardised_unit = resolve_unit_label(
            filename=filename,
            worksheet=sheet_name,
            variable=variable,
            observed_unit=observed_unit,
            expected_unit=expected_unit,
        )

        unit_records.append(
            {
                "original_filename": filename,
                "worksheet": sheet_name,
                "test_type_standard": test_type,
                "variable": variable,
                "observed_unit": observed_unit,
                "expected_unit": expected_unit,
                "standardised_unit": standardised_unit,
                "unit_status": unit_status,
            }
        )

    summary = {
        "original_filename": filename,
        "experiment_id": experiment_id,
        "test_type_standard": test_type,
        "first_measurement_row": (
            measurement_rows[0] if measurement_rows else None
        ),
        "last_measurement_row": (
            measurement_rows[-1] if measurement_rows else None
        ),
        "rows_extracted": len(measurements),
        "audited_row_count": audited_row_count(filename),
        "extra_source_columns": extra_source_columns,
    }
    summary["row_count_match"] = (
        summary["rows_extracted"] == summary["audited_row_count"]
    )

    return measurements, pd.DataFrame(unit_records), summary


# ----------------------------------------------------------------
# 7. Validate the Stage 03 starting state
# ----------------------------------------------------------------

if len(experiment_register) != 20:
    raise ValueError(
        f"Expected 20 registered experiments, found {len(experiment_register)}."
    )

if experiment_register["experiment_id"].nunique() != 20:
    raise ValueError("Experiment IDs are not unique.")

if len(measurement_schema_fields) != 11:
    raise ValueError(
        f"Expected 11 measurement-schema fields, "
        f"found {len(measurement_schema_fields)}."
    )

if len(issue_log) != 8:
    raise ValueError(
        f"Expected 8 Stage 01 QC findings, found {len(issue_log)}."
    )

if len(unit_mapping) != 6:
    raise ValueError(
        f"Expected 6 approved unit-label mapping records, "
        f"found {len(unit_mapping)}."
    )


print("STAGE 03 REPRODUCIBLE SETUP AND INPUT PREFLIGHT")
print("=" * 62)
print(f"Environment:              {environment_name}")
print(f"Python version:           {python_version}")
print(f"NumPy version:            {numpy_version}")
print(f"pandas version:           {pandas_version}")
print(f"openpyxl version:         {openpyxl_version}")
print(f"Project root:             {project_root}")
print(f"Experiment records:       {len(experiment_register)}")
print(f"Unique experiment IDs:    {experiment_register['experiment_id'].nunique()}")
print(f"Measurement schema fields:{len(measurement_schema_fields)}")
print(f"Documented QC issues:     {len(issue_log)}")
print(f"Validated unit mappings:  {len(unit_mapping)}")
print("=" * 62)
print("STAGE 03 SETUP STATUS: PASS")


STAGE 03 REPRODUCIBLE SETUP AND INPUT PREFLIGHT
Environment:              hydrogel-rheology
Python version:           3.11.16
NumPy version:            1.26.4
pandas version:           2.2.3
openpyxl version:         3.1.5
Project root:             C:\Users\sufer\Documents\collagen-hydrogel-rheology-data-curation
Experiment records:       20
Unique experiment IDs:    20
Measurement schema fields:11
Documented QC issues:     8
Validated unit mappings:  6
STAGE 03 SETUP STATUS: PASS


## 1. Source-workbook reconciliation

The Stage 02 experiment register is reconciled against the actual Excel
workbooks in the audited interim-data directory before any measurement values
are processed.

This check answers a simple question:

> Are the 20 workbooks registered during schema design exactly the 20 workbooks
> available for Stage 03 processing?

No file is created or modified by this check.


In [4]:
# ================================================================
# STAGE 03 — SOURCE-WORKBOOK RECONCILIATION
# ================================================================

require_objects("experiment_register", "source_directory")

registered_filenames = set(
    experiment_register["original_filename"]
    .dropna()
    .astype(str)
)

available_filenames = {
    path.name
    for path in source_directory.glob("*.xlsx")
}

missing_workbooks = sorted(
    registered_filenames - available_filenames
)

unexpected_workbooks = sorted(
    available_filenames - registered_filenames
)

if missing_workbooks or unexpected_workbooks:
    raise RuntimeError(
        "SOURCE-WORKBOOK RECONCILIATION FAILED\n\n"
        f"Missing workbooks: {missing_workbooks}\n"
        f"Unexpected workbooks: {unexpected_workbooks}"
    )

print("STAGE 03 SOURCE-WORKBOOK RECONCILIATION")
print("=" * 58)
print(f"Registered workbooks:      {len(registered_filenames)}")
print(f"Available .xlsx workbooks: {len(available_filenames)}")
print(f"Missing workbooks:         {len(missing_workbooks)}")
print(f"Unexpected workbooks:      {len(unexpected_workbooks)}")
print("=" * 58)
print("SOURCE-WORKBOOK RECONCILIATION: PASS")


STAGE 03 SOURCE-WORKBOOK RECONCILIATION
Registered workbooks:      20
Available .xlsx workbooks: 20
Missing workbooks:         0
Unexpected workbooks:      0
SOURCE-WORKBOOK RECONCILIATION: PASS


## 2. Controlled extraction tests

Before processing all 20 workbooks, the shared extraction function is tested on
one frequency-sweep workbook and one time-sweep workbook selected
deterministically from the experiment register.

The tests verify the audited heading sequence, row count, schema order and
identifier uniqueness. For the frequency sweep, the three time-sweep-only
fields must remain structurally missing. For the time sweep, those fields must
be populated.

The known zero-storage-modulus observation is retained; it is not silently
replaced or removed. No processed file is written during these tests.


In [6]:
# ================================================================
# STAGE 03 — CONTROLLED FREQUENCY- AND TIME-SWEEP EXTRACTION TESTS
# ================================================================

require_objects(
    "experiment_register",
    "measurement_schema_fields",
    "extract_registered_experiment",
)


def select_test_experiment(test_type):
    subset = experiment_register.loc[
        experiment_register["test_type_standard"].eq(test_type)
    ]

    if subset.empty:
        raise ValueError(
            f"No registered experiment found for {test_type}."
        )

    return (
        subset.sort_values(
            [
                "collagen_concentration_mg_ml",
                "replicate_id",
                "original_filename",
            ]
        )
        .iloc[0]
    )


frequency_test_experiment = select_test_experiment(
    "frequency_sweep"
)

time_test_experiment = select_test_experiment(
    "time_sweep"
)

frequency_test, frequency_test_units, frequency_test_summary = (
    extract_registered_experiment(frequency_test_experiment)
)

time_test, time_test_units, time_test_summary = (
    extract_registered_experiment(time_test_experiment)
)


# Frequency-sweep validation
if not frequency_test_summary["row_count_match"]:
    raise ValueError("Controlled frequency-sweep row count does not match audit.")

if not frequency_test["measurement_id"].is_unique:
    raise ValueError("Controlled frequency-sweep measurement IDs are not unique.")

time_only_fields = [
    "shear_stress_pa",
    "strain_percent",
    "complex_viscosity_pa_s",
]

frequency_structural_na_rows = int(
    frequency_test[time_only_fields]
    .isna()
    .all(axis=1)
    .sum()
)

if frequency_structural_na_rows != len(frequency_test):
    raise ValueError(
        "Frequency-sweep structural missingness does not match the schema."
    )


# Time-sweep validation
if not time_test_summary["row_count_match"]:
    raise ValueError("Controlled time-sweep row count does not match audit.")

if not time_test["measurement_id"].is_unique:
    raise ValueError("Controlled time-sweep measurement IDs are not unique.")

time_rows_missing_time_fields = int(
    time_test[time_only_fields]
    .isna()
    .any(axis=1)
    .sum()
)

if time_rows_missing_time_fields != 0:
    raise ValueError(
        "Controlled time-sweep extraction contains unexpected missing "
        "values in time-sweep-only fields."
    )


zero_storage_count = int(
    frequency_test["storage_modulus_pa"]
    .eq(0)
    .sum()
)


print("CONTROLLED EXTRACTION TESTS")
print("=" * 62)
print(
    f"Frequency workbook:       "
    f"{frequency_test_experiment['original_filename']}"
)
print(
    f"Frequency rows extracted: "
    f"{len(frequency_test)}"
)
print(
    f"Frequency source rows:     "
    f"{frequency_test['source_excel_row'].min()} to "
    f"{frequency_test['source_excel_row'].max()}"
)
print(
    f"Frequency structural-NA:  "
    f"{frequency_structural_na_rows}"
)
print(
    f"Zero G' values retained:  "
    f"{zero_storage_count}"
)
print("-" * 62)
print(
    f"Time workbook:            "
    f"{time_test_experiment['original_filename']}"
)
print(
    f"Time rows extracted:      "
    f"{len(time_test)}"
)
print(
    f"Time source rows:         "
    f"{time_test['source_excel_row'].min()} to "
    f"{time_test['source_excel_row'].max()}"
)
print(
    f"Time rows missing "
    f"time-only fields:        "
    f"{time_rows_missing_time_fields}"
)
print("=" * 62)
print("CONTROLLED EXTRACTION TESTS: PASS")

display(frequency_test.head(6))
display(time_test.head(6))


CONTROLLED EXTRACTION TESTS
Frequency workbook:       Colageno_bov_0.8_1_frecuencia.xlsx
Frequency rows extracted: 21
Frequency source rows:     6 to 26
Frequency structural-NA:  21
Zero G' values retained:  1
--------------------------------------------------------------
Time workbook:            Colageno_bov_0.8_1_tiempo.xlsx
Time rows extracted:      45
Time source rows:         6 to 50
Time rows missing time-only fields:        0
CONTROLLED EXTRACTION TESTS: PASS


,measurement_id,experiment_id,source_excel_row,measurement_point,angular_frequency_rad_s,storage_modulus_pa,loss_modulus_pa,temperature_c,shear_stress_pa,strain_percent,complex_viscosity_pa_s
0,zenodo_17413651::Colageno_bov_0.8_1_frecuencia...,zenodo_17413651::Colageno_bov_0.8_1_frecuencia...,6,1,0.628,0.000,1.590,37,<NA>,<NA>,<NA>
1,zenodo_17413651::Colageno_bov_0.8_1_frecuencia...,zenodo_17413651::Colageno_bov_0.8_1_frecuencia...,7,2,0.791,0.986,1.633,37,<NA>,<NA>,<NA>
2,zenodo_17413651::Colageno_bov_0.8_1_frecuencia...,zenodo_17413651::Colageno_bov_0.8_1_frecuencia...,8,3,0.996,1.930,1.593,37,<NA>,<NA>,<NA>
3,zenodo_17413651::Colageno_bov_0.8_1_frecuencia...,zenodo_17413651::Colageno_bov_0.8_1_frecuencia...,9,4,1.250,2.260,1.680,37,<NA>,<NA>,<NA>
4,zenodo_17413651::Colageno_bov_0.8_1_frecuencia...,zenodo_17413651::Colageno_bov_0.8_1_frecuencia...,10,5,1.580,1.708,0.843,37,<NA>,<NA>,<NA>
5,zenodo_17413651::Colageno_bov_0.8_1_frecuencia...,zenodo_17413651::Colageno_bov_0.8_1_frecuencia...,11,6,1.990,0.980,1.500,37,<NA>,<NA>,<NA>


,measurement_id,experiment_id,source_excel_row,measurement_point,angular_frequency_rad_s,storage_modulus_pa,loss_modulus_pa,temperature_c,shear_stress_pa,strain_percent,complex_viscosity_pa_s
0,zenodo_17413651::Colageno_bov_0.8_1_tiempo.xls...,zenodo_17413651::Colageno_bov_0.8_1_tiempo.xls...,6,1,6.28,0.50,0.330,37.2,0.01330,1.000,0.211
1,zenodo_17413651::Colageno_bov_0.8_1_tiempo.xls...,zenodo_17413651::Colageno_bov_0.8_1_tiempo.xls...,7,2,6.28,0.60,0.906,37.1,0.00912,0.999,0.145
2,zenodo_17413651::Colageno_bov_0.8_1_tiempo.xls...,zenodo_17413651::Colageno_bov_0.8_1_tiempo.xls...,8,3,6.28,0.70,0.698,37.1,0.00699,1.000,0.111
3,zenodo_17413651::Colageno_bov_0.8_1_tiempo.xls...,zenodo_17413651::Colageno_bov_0.8_1_tiempo.xls...,9,4,6.28,0.93,0.700,37.0,0.03280,0.988,0.528
4,zenodo_17413651::Colageno_bov_0.8_1_tiempo.xls...,zenodo_17413651::Colageno_bov_0.8_1_tiempo.xls...,10,5,6.28,0.50,1.180,37.0,0.01190,1.000,0.188
5,zenodo_17413651::Colageno_bov_0.8_1_tiempo.xls...,zenodo_17413651::Colageno_bov_0.8_1_tiempo.xls...,11,6,6.28,1.08,1.250,37.0,0.01250,1.000,0.199


## 3. Full-dataset extraction and unit-label validation

The validated extraction function is now applied to all 20 registered
workbooks.

For each workbook the workflow:

1. verifies the expected source heading sequence;
2. extracts only the audited source columns;
3. checks the extracted row count against Stage 01;
4. preserves the original Excel row number and measurement-point value;
5. builds deterministic `measurement_id` values;
6. validates source unit labels against the canonical units; and
7. accepts non-canonical unit labels only when they are covered by a previously
   approved Stage 01 mapping.

For example, `[ｰC]` is recognised as the approved source-label variant of
`[°C]`, and `[Paｷs]` as the approved source-label variant of `[Pa·s]`.
These are **label mappings only**; numerical measurements are not changed.

Only the expected source columns are extracted. This means the undocumented
extra formula column identified during Stage 01 remains in its original source
workbook but is not incorporated into the canonical processed table.


In [8]:
# ================================================================
# STAGE 03 — FULL-DATASET EXTRACTION AND UNIT-LABEL VALIDATION
# ================================================================

require_objects(
    "experiment_register",
    "measurement_schema_fields",
    "extract_registered_experiment",
)

all_measurement_records = []
all_unit_records = []
workbook_summary_records = []

ordered_experiments = (
    experiment_register
    .sort_values(
        [
            "collagen_concentration_mg_ml",
            "replicate_id",
            "test_type_standard",
            "original_filename",
        ]
    )
    .reset_index(drop=True)
)

total_workbooks = len(ordered_experiments)

for sequence_number, (_, experiment) in enumerate(
    ordered_experiments.iterrows(),
    start=1,
):
    print(
        f"[{sequence_number:02d}/{total_workbooks:02d}] "
        f"{experiment['original_filename']}",
        flush=True,
    )

    measurements, unit_records, summary = (
        extract_registered_experiment(experiment)
    )

    if not summary["row_count_match"]:
        raise ValueError(
            f"Row-count mismatch in {summary['original_filename']}: "
            f"expected {summary['audited_row_count']}, "
            f"found {summary['rows_extracted']}."
        )

    all_measurement_records.extend(
        measurements.to_dict(orient="records")
    )
    all_unit_records.extend(
        unit_records.to_dict(orient="records")
    )
    workbook_summary_records.append(summary)


all_measurements = (
    pd.DataFrame(all_measurement_records)
    .reindex(columns=measurement_schema_fields)
)

unit_standardisation_validation = pd.DataFrame(
    all_unit_records
)

workbook_extraction_summary = pd.DataFrame(
    workbook_summary_records
)


# ----------------------------------------------------------------
# Full-dataset structural checks
# ----------------------------------------------------------------

if len(all_measurements) != 660:
    raise ValueError(
        f"Expected 660 measurement rows, found {len(all_measurements)}."
    )

if all_measurements["measurement_id"].nunique() != 660:
    raise ValueError("Measurement IDs are not globally unique.")

if all_measurements["measurement_id"].duplicated().any():
    raise ValueError("Duplicate measurement IDs were detected.")

if all_measurements.columns.tolist() != measurement_schema_fields:
    raise ValueError(
        "Full measurement table does not match the Stage 02 schema order."
    )

if workbook_extraction_summary["row_count_match"].eq(False).any():
    raise ValueError("One or more workbook row counts do not match Stage 01.")


# ----------------------------------------------------------------
# Unit-label checks
# ----------------------------------------------------------------

unit_bearing_records = unit_standardisation_validation.loc[
    unit_standardisation_validation["expected_unit"].notna()
]

canonical_unit_count = int(
    unit_bearing_records["unit_status"]
    .eq("canonical")
    .sum()
)

approved_mapping_count = int(
    unit_bearing_records["unit_status"]
    .eq("approved_mapping")
    .sum()
)

unresolved_unit_count = int(
    unit_bearing_records["unit_status"]
    .eq("unresolved")
    .sum()
)

unexpected_unitless_count = int(
    unit_standardisation_validation["unit_status"]
    .eq("unexpected_unit_label")
    .sum()
)

if unresolved_unit_count != 0:
    unresolved = unit_standardisation_validation.loc[
        unit_standardisation_validation["unit_status"].eq("unresolved")
    ]
    display(unresolved)
    raise ValueError(
        "One or more source unit labels are not covered by the "
        "approved Stage 01 mapping rules."
    )

if unexpected_unitless_count != 0:
    unexpected = unit_standardisation_validation.loc[
        unit_standardisation_validation[
            "unit_status"
        ].eq("unexpected_unit_label")
    ]
    display(unexpected)
    raise ValueError(
        "An unexpected unit label was found on a field that should be unitless."
    )


frequency_experiment_ids = set(
    experiment_register.loc[
        experiment_register["test_type_standard"].eq("frequency_sweep"),
        "experiment_id",
    ].astype(str)
)

time_experiment_ids = set(
    experiment_register.loc[
        experiment_register["test_type_standard"].eq("time_sweep"),
        "experiment_id",
    ].astype(str)
)

frequency_rows = int(
    all_measurements["experiment_id"]
    .astype(str)
    .isin(frequency_experiment_ids)
    .sum()
)

time_rows = int(
    all_measurements["experiment_id"]
    .astype(str)
    .isin(time_experiment_ids)
    .sum()
)

workbooks_with_extra_source_columns = int(
    workbook_extraction_summary["extra_source_columns"]
    .gt(0)
    .sum()
)


print()
print("FULL-DATASET EXTRACTION AND UNIT-LABEL VALIDATION")
print("=" * 70)
print(f"Workbooks processed:                  {total_workbooks}")
print(f"Experiments represented:              {all_measurements['experiment_id'].nunique()}")
print(f"Rows extracted:                       {len(all_measurements)}")
print(f"Unique measurement IDs:               {all_measurements['measurement_id'].nunique()}")
print(f"Frequency-sweep rows:                 {frequency_rows}")
print(f"Time-sweep rows:                      {time_rows}")
print(f"Workbook row-count mismatches:        {int((~workbook_extraction_summary['row_count_match']).sum())}")
print(f"Workbooks with extra source columns:  {workbooks_with_extra_source_columns}")
print("-" * 70)
print(f"Unit-bearing fields checked:          {len(unit_bearing_records)}")
print(f"Already canonical unit labels:        {canonical_unit_count}")
print(f"Approved mappings recognised:         {approved_mapping_count}")
print(f"Unresolved unit labels:               {unresolved_unit_count}")
print(f"Unexpected units on unitless fields:  {unexpected_unitless_count}")
print("Numerical measurements modified:      0")
print("Source workbooks modified:            0")
print("Files written:                        0")
print("=" * 70)
print("FULL-DATASET EXTRACTION AND UNIT-LABEL VALIDATION: PASS")

display(
    workbook_extraction_summary[
        [
            "original_filename",
            "test_type_standard",
            "rows_extracted",
            "audited_row_count",
            "row_count_match",
            "extra_source_columns",
        ]
    ]
)

mapped_unit_labels = (
    unit_standardisation_validation.loc[
        unit_standardisation_validation["unit_status"].eq(
            "approved_mapping"
        ),
        [
            "original_filename",
            "worksheet",
            "variable",
            "observed_unit",
            "standardised_unit",
        ],
    ]
    .reset_index(drop=True)
)

if not mapped_unit_labels.empty:
    print("\nAPPROVED SOURCE UNIT-LABEL MAPPINGS")
    display(mapped_unit_labels)


[01/20] Colageno_bov_0.8_1_frecuencia.xlsx
[02/20] Colageno_bov_0.8_1_tiempo.xlsx
[03/20] Colageno_bov_0.8_2_frecuencia.xlsx
[04/20] Colageno_bov_0.8_2_tiempo.xlsx
[05/20] Colageno_bov_0.8_3_frecuencia.xlsx
[06/20] Colageno_bov_0.8_3_tiempo.xlsx
[07/20] Colageno_bov_0.8_4_frecuencia.xlsx
[08/20] Colageno_bov_0.8_4_tiempo.xlsx
[09/20] Colageno_bov_1.5_1_frecuencia.xlsx
[10/20] Colageno_bov_1.5_1_tiempo.xlsx
[11/20] Colageno_bov_1.5_2_frecuencia.xlsx
[12/20] Colageno_bov_1.5_2_tiempo.xlsx
[13/20] Colageno_bov_1.5_3_frecuencia.xlsx
[14/20] Colageno_bov_1.5_3_tiempo.xlsx
[15/20] Colageno_bov_2.3_1_frecuencia.xlsx
[16/20] Colageno_bov_2.3_1_tiempo.xlsx
[17/20] Colageno_bov_2.3_2_frecuencia.xlsx
[18/20] Colageno_bov_2.3_2_tiempo.xlsx
[19/20] Colageno_bov_2.3_3_frecuencia.xlsx
[20/20] Colageno_bov_2.3_3_tiempo.xlsx

FULL-DATASET EXTRACTION AND UNIT-LABEL VALIDATION
Workbooks processed:                  20
Experiments represented:              20
Rows extracted:                       660
Uniqu

,original_filename,test_type_standard,rows_extracted,audited_row_count,row_count_match,extra_source_columns
0,Colageno_bov_0.8_1_frecuencia.xlsx,frequency_sweep,21,21,True,0
1,Colageno_bov_0.8_1_tiempo.xlsx,time_sweep,45,45,True,0
2,Colageno_bov_0.8_2_frecuencia.xlsx,frequency_sweep,21,21,True,1
3,Colageno_bov_0.8_2_tiempo.xlsx,time_sweep,45,45,True,0
4,Colageno_bov_0.8_3_frecuencia.xlsx,frequency_sweep,21,21,True,0
5,Colageno_bov_0.8_3_tiempo.xlsx,time_sweep,45,45,True,0
6,Colageno_bov_0.8_4_frecuencia.xlsx,frequency_sweep,21,21,True,0
7,Colageno_bov_0.8_4_tiempo.xlsx,time_sweep,45,45,True,0
8,Colageno_bov_1.5_1_frecuencia.xlsx,frequency_sweep,21,21,True,0
9,Colageno_bov_1.5_1_tiempo.xlsx,time_sweep,45,45,True,0



APPROVED SOURCE UNIT-LABEL MAPPINGS


,original_filename,worksheet,variable,observed_unit,standardised_unit
0,Colageno_bov_1.5_1_frecuencia.xlsx,Hoja1,Temperature,[ｰC],[°C]
1,Colageno_bov_1.5_1_tiempo.xlsx,Hoja1,Complex Viscosity,[Paｷs],[Pa·s]
2,Colageno_bov_1.5_1_tiempo.xlsx,Hoja1,Temperature,[ｰC],[°C]
3,Colageno_bov_2.3_2_frecuencia.xlsx,Hoja1,Temperature,[ｰC],[°C]
4,Colageno_bov_2.3_2_tiempo.xlsx,Hoja1,Complex Viscosity,[Paｷs],[Pa·s]
5,Colageno_bov_2.3_2_tiempo.xlsx,Hoja1,Temperature,[ｰC],[°C]


## 4. Measurement-level quality-control linkage

Stage 01 recorded eight quality-control findings. Most are workbook- or
unit-label-level findings; only findings that identify a single measurement
cell can be linked directly to a processed row.

The documented zero-storage-modulus finding is linked using:

- original filename;
- worksheet;
- source Excel row; and
- the deterministic `measurement_id`.

The source value remains unchanged. This step adds traceability; it does not
correct the measurement.


In [10]:
# ================================================================
# STAGE 03 — MEASUREMENT-LEVEL QC LINKAGE
# ================================================================

require_objects(
    "issue_log",
    "experiment_register",
    "all_measurements",
)


def extract_single_excel_row(cell_reference):
    """Return an Excel row number for a single-cell reference such as C6."""
    if not isinstance(cell_reference, str):
        return None

    match = re.fullmatch(
        r"[A-Za-z]+(\d+)",
        cell_reference.strip(),
    )

    return int(match.group(1)) if match else None


qc_scope = issue_log.copy()

qc_scope["source_excel_row"] = (
    qc_scope["cell_or_range"]
    .apply(extract_single_excel_row)
)

measurement_level_issues = (
    qc_scope.loc[
        qc_scope["source_excel_row"].notna()
        & qc_scope["source_excel_row"].ge(6)
    ]
    .copy()
)

non_measurement_level_issues = (
    qc_scope.loc[
        ~qc_scope.index.isin(measurement_level_issues.index)
    ]
    .copy()
)

# Stage 01 contains one measurement-level finding in this dataset.
qc_variable_to_measurement_field = {
    "Storage Modulus (G′)": "storage_modulus_pa",
}

measurement_qc_links = []

for _, issue in measurement_level_issues.iterrows():
    filename = issue["original_filename"]
    worksheet = issue["worksheet"]
    source_excel_row = int(issue["source_excel_row"])

    experiment_match = experiment_register.loc[
        experiment_register["original_filename"].eq(filename)
        & experiment_register["sheet_name"].eq(worksheet)
    ]

    if len(experiment_match) != 1:
        raise ValueError(
            f"Expected exactly one experiment for "
            f"{filename} / {worksheet}, "
            f"found {len(experiment_match)}."
        )

    experiment_id = experiment_match["experiment_id"].iloc[0]

    measurement_id = (
        f"{experiment_id}::row_{source_excel_row}"
    )

    measurement_match = all_measurements.loc[
        all_measurements["measurement_id"].eq(measurement_id)
    ]

    if len(measurement_match) != 1:
        raise ValueError(
            f"Expected exactly one processed measurement for "
            f"{measurement_id}, found {len(measurement_match)}."
        )

    canonical_field = qc_variable_to_measurement_field.get(
        issue["variable"]
    )

    if canonical_field is None:
        raise ValueError(
            "No canonical measurement-field mapping is defined for "
            f"QC variable '{issue['variable']}'."
        )

    extracted_value = measurement_match[
        canonical_field
    ].iloc[0]

    measurement_qc_links.append(
        {
            "measurement_id": measurement_id,
            "issue_id": issue["issue_id"],
            "original_filename": filename,
            "worksheet": worksheet,
            "source_excel_row": source_excel_row,
            "variable": issue["variable"],
            "canonical_field": canonical_field,
            "extracted_value": extracted_value,
            "issue_category": issue["issue_category"],
            "resolution_status": issue["resolution_status"],
            "processing_decision": issue["processing_decision"],
        }
    )


measurement_qc_linkage = pd.DataFrame(
    measurement_qc_links
)

if len(measurement_qc_linkage) != len(measurement_level_issues):
    raise ValueError(
        "Not every measurement-level QC issue was linked exactly once."
    )

value_001 = measurement_qc_linkage.loc[
    measurement_qc_linkage["issue_id"].eq("VALUE-001")
]

if len(value_001) != 1:
    raise ValueError("VALUE-001 was not linked exactly once.")

if float(value_001["extracted_value"].iloc[0]) != 0.0:
    raise ValueError(
        "VALUE-001 no longer corresponds to G' = 0 Pa."
    )


print("MEASUREMENT-LEVEL QC LINKAGE")
print("=" * 62)
print(f"Total Stage 01 QC findings:              {len(issue_log)}")
print(f"Measurement-level QC findings:           {len(measurement_level_issues)}")
print(f"Non-measurement-level QC findings:       {len(non_measurement_level_issues)}")
print(f"Measurement issues successfully linked:  {len(measurement_qc_linkage)}")
print(f"VALUE-001 extracted G' value:             {value_001['extracted_value'].iloc[0]} Pa")
print("Files written:                           0")
print("=" * 62)
print("MEASUREMENT-LEVEL QC LINKAGE: PASS")

display(measurement_qc_linkage)


MEASUREMENT-LEVEL QC LINKAGE
Total Stage 01 QC findings:              8
Measurement-level QC findings:           1
Non-measurement-level QC findings:       7
Measurement issues successfully linked:  1
VALUE-001 extracted G' value:             0.0 Pa
Files written:                           0
MEASUREMENT-LEVEL QC LINKAGE: PASS


,measurement_id,issue_id,original_filename,worksheet,source_excel_row,variable,canonical_field,extracted_value,issue_category,resolution_status,processing_decision
0,zenodo_17413651::Colageno_bov_0.8_1_frecuencia...,VALUE-001,Colageno_bov_0.8_1_frecuencia.xlsx,Hoja1,6,Storage Modulus (G′),storage_modulus_pa,0.0,Zero rheological value,Retained with quality-control flag,"Retain G′ = 0 Pa, flag the observation, and re..."


## 5. Final in-memory measurement-table validation

The complete canonical table is now checked as a single dataset before any
processed CSV is written.

The implemented checks cover:

- exact Stage 02 schema membership and order;
- total row count and experiment coverage;
- deterministic identifier construction;
- duplicate identifiers and duplicate experiment/source-row pairs;
- valid source-row and measurement-point values;
- numeric compatibility of populated rheological fields;
- frequency-sweep structural missingness;
- completeness of time-sweep-only fields;
- workbook-level row counts; and
- retention of the documented zero-storage-modulus observation.

These are structural, provenance and data-type checks. They do not establish
that every rheological measurement is physically plausible or scientifically
correct.


In [12]:
# ================================================================
# STAGE 03 — FINAL IN-MEMORY MEASUREMENT-TABLE VALIDATION
# ================================================================

require_objects(
    "all_measurements",
    "measurement_schema_fields",
    "experiment_register",
    "workbook_extraction_summary",
    "measurement_qc_linkage",
)

if all_measurements.columns.tolist() != measurement_schema_fields:
    raise ValueError(
        "The final table does not match the Stage 02 schema order."
    )

if len(all_measurements) != 660:
    raise ValueError(
        f"Expected 660 rows, found {len(all_measurements)}."
    )

if all_measurements["experiment_id"].nunique() != 20:
    raise ValueError(
        "The final table does not represent all 20 registered experiments."
    )


# Identifier construction
expected_measurement_ids = (
    all_measurements["experiment_id"].astype(str)
    + "::row_"
    + all_measurements["source_excel_row"].astype(int).astype(str)
)

measurement_id_rule_mismatches = int(
    (
        all_measurements["measurement_id"].astype(str)
        != expected_measurement_ids
    ).sum()
)

duplicate_measurement_ids = int(
    all_measurements["measurement_id"]
    .duplicated()
    .sum()
)

duplicate_experiment_source_rows = int(
    all_measurements[
        ["experiment_id", "source_excel_row"]
    ]
    .duplicated()
    .sum()
)


# Provenance fields
source_rows_numeric = pd.to_numeric(
    all_measurements["source_excel_row"],
    errors="coerce",
)

measurement_points_numeric = pd.to_numeric(
    all_measurements["measurement_point"],
    errors="coerce",
)

invalid_source_rows = int(
    (
        source_rows_numeric.isna()
        | source_rows_numeric.lt(6)
    ).sum()
)

invalid_measurement_points = int(
    (
        measurement_points_numeric.isna()
        | measurement_points_numeric.lt(1)
    ).sum()
)


# Rheology-field numeric compatibility
rheology_fields = [
    "angular_frequency_rad_s",
    "storage_modulus_pa",
    "loss_modulus_pa",
    "temperature_c",
    "shear_stress_pa",
    "strain_percent",
    "complex_viscosity_pa_s",
]

non_numeric_populated_values = 0

for field in rheology_fields:
    populated = all_measurements[field].notna()

    converted = pd.to_numeric(
        all_measurements.loc[populated, field],
        errors="coerce",
    )

    non_numeric_populated_values += int(
        converted.isna().sum()
    )


# Structural missingness by test type
frequency_mask = (
    all_measurements["experiment_id"]
    .astype(str)
    .isin(frequency_experiment_ids)
)

time_mask = (
    all_measurements["experiment_id"]
    .astype(str)
    .isin(time_experiment_ids)
)

frequency_structural_na_rows = int(
    all_measurements.loc[
        frequency_mask,
        time_only_fields,
    ]
    .isna()
    .all(axis=1)
    .sum()
)

time_rows_missing_time_fields = int(
    all_measurements.loc[
        time_mask,
        time_only_fields,
    ]
    .isna()
    .any(axis=1)
    .sum()
)


# Workbook-level row counts
workbook_row_count_mismatches = int(
    (~workbook_extraction_summary["row_count_match"])
    .sum()
)

zero_storage_values = int(
    all_measurements["storage_modulus_pa"]
    .eq(0)
    .sum()
)


validation_failures = {
    "measurement-ID rule mismatches": measurement_id_rule_mismatches,
    "duplicate measurement IDs": duplicate_measurement_ids,
    "duplicate experiment/source-row pairs": duplicate_experiment_source_rows,
    "invalid source Excel rows": invalid_source_rows,
    "invalid measurement points": invalid_measurement_points,
    "non-numeric populated rheology values": non_numeric_populated_values,
    "workbook row-count mismatches": workbook_row_count_mismatches,
}

nonzero_failures = {
    label: count
    for label, count in validation_failures.items()
    if count != 0
}

if nonzero_failures:
    raise ValueError(
        "FINAL IN-MEMORY VALIDATION FAILED\n\n"
        + "\n".join(
            f"- {label}: {count}"
            for label, count in nonzero_failures.items()
        )
    )

if frequency_structural_na_rows != 210:
    raise ValueError(
        "Frequency-sweep structural missingness does not match "
        "the expected 210 rows."
    )

if time_rows_missing_time_fields != 0:
    raise ValueError(
        "Time-sweep data contain unexpected missing values in "
        "time-sweep-only fields."
    )

if zero_storage_values != 1:
    raise ValueError(
        f"Expected one retained G' = 0 observation, "
        f"found {zero_storage_values}."
    )


print("FINAL IN-MEMORY MEASUREMENT-TABLE VALIDATION")
print("=" * 70)
print(f"Rows validated:                         {len(all_measurements)}")
print(f"Schema fields:                          {len(all_measurements.columns)}")
print(f"Registered experiments represented:     {all_measurements['experiment_id'].nunique()}")
print(f"Measurement-ID rule mismatches:         {measurement_id_rule_mismatches}")
print(f"Duplicate measurement IDs:              {duplicate_measurement_ids}")
print(f"Duplicate experiment/source-row pairs:  {duplicate_experiment_source_rows}")
print(f"Invalid source Excel rows:               {invalid_source_rows}")
print(f"Invalid measurement points:              {invalid_measurement_points}")
print(f"Non-numeric populated rheology values:   {non_numeric_populated_values}")
print(f"Frequency structural-NA rows:            {frequency_structural_na_rows}")
print(f"Time rows missing time-only fields:      {time_rows_missing_time_fields}")
print(f"Workbook row-count mismatches:           {workbook_row_count_mismatches}")
print(f"Retained G' = 0 observations:            {zero_storage_values}")
print("Files written:                           0")
print("=" * 70)
print("FINAL IN-MEMORY MEASUREMENT-TABLE VALIDATION: PASS")


FINAL IN-MEMORY MEASUREMENT-TABLE VALIDATION
Rows validated:                         660
Schema fields:                          11
Registered experiments represented:     20
Measurement-ID rule mismatches:         0
Duplicate measurement IDs:              0
Duplicate experiment/source-row pairs:  0
Invalid source Excel rows:               0
Invalid measurement points:              0
Non-numeric populated rheology values:   0
Frequency structural-NA rows:            210
Time rows missing time-only fields:      0
Workbook row-count mismatches:           0
Retained G' = 0 observations:            1
Files written:                           0
FINAL IN-MEMORY MEASUREMENT-TABLE VALIDATION: PASS


## 6. Repository publication-safety check

The source archive, extracted workbooks and row-level processed measurements are
kept out of the public repository.

This notebook **verifies** the repository-level `.gitignore`; it does not create,
overwrite or modify that file. The required exclusions are:

- `data/raw/**`
- `data/interim/**`
- `data/processed/**`

Optional `.gitkeep` files may remain tracked so that the directory structure can
be represented without publishing the protected data.

> **Important:** `.gitignore` protects normal Git workflows. It does not prevent
> someone from manually uploading an ignored data file through the GitHub web
> interface.


In [14]:
# ================================================================
# STAGE 03 — REPOSITORY PUBLICATION-SAFETY CHECK
# Read-only and restart-safe
# ================================================================

require_objects("project_root")

gitignore_file = project_root / ".gitignore"

if not gitignore_file.exists():
    raise FileNotFoundError(
        f".gitignore was not found at the project root:\n"
        f"{gitignore_file}\n\n"
        "Create the repository .gitignore before exporting "
        "row-level processed data."
    )

gitignore_text = gitignore_file.read_text(
    encoding="utf-8"
)

required_protection_rules = [
    "data/raw/**",
    "data/interim/**",
    "data/processed/**",
]

missing_protection_rules = [
    rule
    for rule in required_protection_rules
    if rule not in gitignore_text
]

if missing_protection_rules:
    raise RuntimeError(
        "Repository publication protection is incomplete.\n\n"
        "Missing .gitignore rule(s):\n"
        + "\n".join(
            f"- {rule}"
            for rule in missing_protection_rules
        )
    )

placeholder_rules = [
    "!data/raw/.gitkeep",
    "!data/interim/.gitkeep",
    "!data/processed/.gitkeep",
]

placeholder_status = {
    rule: rule in gitignore_text
    for rule in placeholder_rules
}


print("REPOSITORY PUBLICATION-SAFETY CHECK")
print("=" * 66)
print(f"Project root:                       {project_root}")
print(f".gitignore found:                   {gitignore_file.exists()}")
print(f"Raw data protected:                 {'data/raw/**' in gitignore_text}")
print(f"Interim data protected:             {'data/interim/**' in gitignore_text}")
print(f"Processed row-level data protected: {'data/processed/**' in gitignore_text}")
print(f"Raw .gitkeep allowed:               {placeholder_status['!data/raw/.gitkeep']}")
print(f"Interim .gitkeep allowed:           {placeholder_status['!data/interim/.gitkeep']}")
print(f"Processed .gitkeep allowed:         {placeholder_status['!data/processed/.gitkeep']}")
print("Files modified or written:          0")
print("=" * 66)
print("REPOSITORY PUBLICATION-SAFETY CHECK: PASS")


REPOSITORY PUBLICATION-SAFETY CHECK
Project root:                       C:\Users\sufer\Documents\collagen-hydrogel-rheology-data-curation
.gitignore found:                   True
Raw data protected:                 True
Interim data protected:             True
Processed row-level data protected: True
Raw .gitkeep allowed:               True
Interim .gitkeep allowed:           True
Processed .gitkeep allowed:         True
Files modified or written:          0
REPOSITORY PUBLICATION-SAFETY CHECK: PASS


## 7. Local processed-measurement export and disk verification

Only after the in-memory table and publication-safety checks pass are the
canonical measurements written to the local `data/processed/` directory.

Two files are produced:

- `frequency_sweep_standardised.csv`
- `time_sweep_standardised.csv`

Both files retain the complete 11-field canonical schema. The three
time-sweep-only fields remain missing in every frequency-sweep record.

The files are reloaded immediately after writing to verify row counts, schema
order, identifier uniqueness and expected structural missingness.

Because `data/processed/**` is excluded by `.gitignore`, these row-level files
are intended to remain local unless reuse and publication rights are confirmed.


In [16]:
# ================================================================
# STAGE 03 — LOCAL PROCESSED-MEASUREMENT EXPORT AND VERIFICATION
# Deterministic and restart-safe
# ================================================================

require_objects(
    "project_root",
    "all_measurements",
    "measurement_schema_fields",
    "frequency_experiment_ids",
    "time_experiment_ids",
)

processed_directory = (
    project_root
    / "data"
    / "processed"
)

processed_directory.mkdir(
    parents=True,
    exist_ok=True,
)

frequency_measurements = (
    all_measurements.loc[
        all_measurements["experiment_id"]
        .astype(str)
        .isin(frequency_experiment_ids)
    ]
    .copy()
    .reindex(columns=measurement_schema_fields)
    .reset_index(drop=True)
)

time_measurements = (
    all_measurements.loc[
        all_measurements["experiment_id"]
        .astype(str)
        .isin(time_experiment_ids)
    ]
    .copy()
    .reindex(columns=measurement_schema_fields)
    .reset_index(drop=True)
)

if len(frequency_measurements) != 210:
    raise ValueError(
        f"Expected 210 frequency-sweep rows, "
        f"found {len(frequency_measurements)}."
    )

if len(time_measurements) != 450:
    raise ValueError(
        f"Expected 450 time-sweep rows, "
        f"found {len(time_measurements)}."
    )

frequency_output_file = (
    processed_directory
    / "frequency_sweep_standardised.csv"
)

time_output_file = (
    processed_directory
    / "time_sweep_standardised.csv"
)

frequency_measurements.to_csv(
    frequency_output_file,
    index=False,
    encoding="utf-8-sig",
)

time_measurements.to_csv(
    time_output_file,
    index=False,
    encoding="utf-8-sig",
)

saved_frequency = pd.read_csv(
    frequency_output_file,
    encoding="utf-8-sig",
    dtype={
        "measurement_id": "string",
        "experiment_id": "string",
    },
)

saved_time = pd.read_csv(
    time_output_file,
    encoding="utf-8-sig",
    dtype={
        "measurement_id": "string",
        "experiment_id": "string",
    },
)

if saved_frequency.columns.tolist() != measurement_schema_fields:
    raise ValueError(
        "Reloaded frequency-sweep file does not match the Stage 02 schema."
    )

if saved_time.columns.tolist() != measurement_schema_fields:
    raise ValueError(
        "Reloaded time-sweep file does not match the Stage 02 schema."
    )

if len(saved_frequency) != 210 or len(saved_time) != 450:
    raise ValueError(
        "Reloaded processed files do not contain the expected row counts."
    )

frequency_duplicate_ids = int(
    saved_frequency["measurement_id"]
    .duplicated()
    .sum()
)

time_duplicate_ids = int(
    saved_time["measurement_id"]
    .duplicated()
    .sum()
)

saved_frequency_structural_na_rows = int(
    saved_frequency[time_only_fields]
    .isna()
    .all(axis=1)
    .sum()
)

saved_time_missing_time_fields = int(
    saved_time[time_only_fields]
    .isna()
    .any(axis=1)
    .sum()
)

if frequency_duplicate_ids != 0 or time_duplicate_ids != 0:
    raise ValueError(
        "Duplicate measurement IDs were found after disk reload."
    )

if saved_frequency_structural_na_rows != 210:
    raise ValueError(
        "Reloaded frequency-sweep file does not preserve "
        "expected structural missingness."
    )

if saved_time_missing_time_fields != 0:
    raise ValueError(
        "Reloaded time-sweep file contains unexpected missing "
        "time-sweep-only fields."
    )


print("LOCAL PROCESSED-MEASUREMENT EXPORT AND VERIFICATION")
print("=" * 72)
print(f"Frequency-sweep rows written:           {len(saved_frequency)}")
print(f"Time-sweep rows written:                {len(saved_time)}")
print(f"Total rows written:                     {len(saved_frequency) + len(saved_time)}")
print(f"Fields in frequency file:               {len(saved_frequency.columns)}")
print(f"Fields in time file:                    {len(saved_time.columns)}")
print(f"Frequency duplicate measurement IDs:    {frequency_duplicate_ids}")
print(f"Time duplicate measurement IDs:         {time_duplicate_ids}")
print(f"Frequency structural-NA rows:            {saved_frequency_structural_na_rows}")
print(f"Time rows missing time-only fields:      {saved_time_missing_time_fields}")
print(f"Frequency output:                        {frequency_output_file}")
print(f"Time output:                             {time_output_file}")
print("=" * 72)
print("LOCAL PROCESSED-MEASUREMENT EXPORT AND VERIFICATION: PASS")


LOCAL PROCESSED-MEASUREMENT EXPORT AND VERIFICATION
Frequency-sweep rows written:           210
Time-sweep rows written:                450
Total rows written:                     660
Fields in frequency file:               11
Fields in time file:                    11
Frequency duplicate measurement IDs:    0
Time duplicate measurement IDs:         0
Frequency structural-NA rows:            210
Time rows missing time-only fields:      0
Frequency output:                        C:\Users\sufer\Documents\collagen-hydrogel-rheology-data-curation\data\processed\frequency_sweep_standardised.csv
Time output:                             C:\Users\sufer\Documents\collagen-hydrogel-rheology-data-curation\data\processed\time_sweep_standardised.csv
LOCAL PROCESSED-MEASUREMENT EXPORT AND VERIFICATION: PASS


## Stage 03 completion status

When every code section above reports `PASS`, Stage 03 has completed the
implemented extraction, unit-label validation, row-level QC linkage, structural
validation and local export workflow.

The resulting row-level processed CSV files remain local and are excluded from
the public repository. The notebook, metadata tables, data dictionaries,
quality-control logs and reproducibility files can be used to document the
curation workflow on GitHub.

### Scientific boundaries carried forward

- Physical sample linkage between frequency- and time-sweep workbooks remains
  unresolved.
- `Meas. Pts.` in the time-sweep workbooks is not treated as elapsed time.
- No unrecorded sample identity, time axis or experimental condition is inferred.
- Stage 03 validates structure, provenance and documented QC behaviour; it does
  not establish the physical plausibility of every measurement.

The next processing stage can therefore focus on exploratory rheology analysis
using the validated local processed tables while preserving these constraints.
